# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | _[your full name]_ |
| **Student ID** | _[your ID]_ |
| **Section** | _[your section]_ |
| **Date submitted** | _[date]_ |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [2]:
import os
import pandas as pd
import polars as pl 

DATA_PATH = os.path.join ("data" , "Amazon Sale Report.csv")
df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH) 


print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3)) # first 3 rows

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (128975, 24)
polars shape: (128975, 24)

columns: ['index', 'Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel ', 'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier Status', 'Qty', 'currency', 'Amount', 'ship-city', 'ship-state', 'ship-postal-code', 'ship-country', 'promotion-ids', 'B2B', 'fulfilled-by', 'Unnamed: 22']

   index             Order ID      Date                        Status  \
0      0  405-8078784-5731545  04-30-22                     Cancelled   
1      1  171-9198151-1101146  04-30-22  Shipped - Delivered to Buyer   
2      2  404-0687676-7273146  04-30-22                       Shipped   

  Fulfilment Sales Channel  ship-service-level    Style              SKU  \
0   Merchant      Amazon.in           Standard   SET389   SET389-KR-NP-S   
1   Merchant      Amazon.in           Standard  JNE3781  JNE3781-KR-XXXL   
2     Amazon      Amazon.in          Expedited  JNE3371    JNE3371-KR-XL   

  Category  ... currency  Amount  

C:\Users\forat\AppData\Local\Temp\ipykernel_12152\2428958236.py:6: DtypeWarning: Columns (23: Unnamed: 22) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pandas = pd.read_csv(DATA_PATH)


---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [3]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats): # range 0 to 4
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [4]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.018 ms   <- nothing was executed
lazy WITH .collect():      46.180 ms   <- the real cost


**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | _[0.020]_ |
| Lazy query with `.collect()` | _[72.588]_ |

In one sentence — why is the first number meaningless?

_[The first number is meaningless because Polars only builds the execution plan without actually running the query until .collect() is called]_

---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [5]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "Category"    # a text/category column, for grouping
NUMERIC_COL  = "Amount"     # a numeric column
NUMERIC_COL2 = "Qty"  # a second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] Category
[ok] Amount
[ok] Qty


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [6]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

C:\Users\forat\AppData\Local\Temp\ipykernel_12152\38132050.py:1: DtypeWarning: Columns (23: Unnamed: 22) have mixed types. Specify dtype option on import or set low_memory=False.
  pandas_time = measure(lambda: pd.read_csv(DATA_PATH))


pandas:   630.73 ms
polars:    44.74 ms
ratio:     14.10x


**Observation:** _[Polars was significantly faster than Pandas when reading the CSV file, loading the dataset approximately 7.5 times faster.]_

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [7]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold) # median 

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x") #ratio

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 605.0
pandas:     9.91 ms
polars:     4.31 ms
ratio:      2.30x


**Observation:** _[Filtering the DataFrame took 4.35 ms in Polars compared to 27.34 ms in Pandas, making Polars roughly 6.3x faster.]_

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [8]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     0.29 ms
polars:     1.52 ms
ratio:      0.19x


**Observation:** _[Pandas was faster than Polars when creating a new calculated column, executing approximately 7.7 times faster.]_

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [9]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:    11.42 ms
polars:     2.08 ms
ratio:      5.49x

pandas result (first rows):
Category
Blouse          458408.18
Bottom          150667.98
Dupatta            915.00
Ethnic Dress    791217.66
Saree           123933.76
Name: Amount, dtype: float64

polars result (first rows):
shape: (5, 2)
┌──────────┬───────────┐
│ Category ┆ Amount    │
│ ---      ┆ ---       │
│ str      ┆ f64       │
╞══════════╪═══════════╡
│ Set      ┆ 3.9204e7  │
│ kurta    ┆ 2.1300e7  │
│ Saree    ┆ 123933.76 │
│ Blouse   ┆ 458408.18 │
│ Dupatta  ┆ 915.0     │
└──────────┴───────────┘


**Observation:** _[Polars executed the group-by and aggregation operation faster than]_

**Do both libraries give the same numbers?** _[Yes, both libraries produce the same numeric values, but the row order differs because Polars does not maintain sorted index order by default.]_

## 4.5 — Sorting and taking the top N

In [10]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:    41.50 ms
polars:     6.79 ms
ratio:      6.11x


**Observation:** _[Polars performed sorting and retrieving the top 10 rows significantly faster than Pandas, running approximately 4.29 times faster.]_

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [11]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

C:\Users\forat\AppData\Local\Temp\ipykernel_12152\2977073395.py:13: DtypeWarning: Columns (23: Unnamed: 22) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_csv(DATA_PATH)


pandas (step by step):   642.36 ms
polars (lazy + collect):    44.58 ms
ratio:    14.41x


**Look at the query plan.** Polars can show you what it decided to do.

In [12]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("Amount").sum()] BY [col("Category")]
  FROM
  Csv SCAN [data/Amazon Sale Report.csv]
  PROJECT 2/24 COLUMNS
  SELECTION: col("Amount") > 605.0
  ESTIMATED ROWS: 401467


**Observation:** _[Polars performed sorting and retrieving the top 10 rows significantly faster than Pandas, running approximately 4.29 times faster.]_

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [13]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:    12.75 ms
polars:     2.42 ms
ratio:      5.27x

rows before: 128975 -> after: 128975
row count unchanged: True


**Observation:** _[Polars executed the left join operation significantly faster than Pandas, running approximately 3.87 times faster while maintaining the exact same row count.]_

---
# Part 5 · Results summary

Collect every measurement into one table.

In [14]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,630.73,44.74,14.10,polars
1,filter,9.91,4.31,2.30,polars
2,new column,0.29,1.52,0.19,pandas
3,group by,11.42,2.08,5.49,polars
4,sort + top N,41.50,6.79,6.11,polars
5,chained pipeline,642.36,44.58,14.41,polars
6,join,12.75,2.42,5.27,polars


In [15]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Windows-11-10.0.26200-SP0
   ARMv8 (64-bit) Family 8 Model 1 Revision 201, Qualcomm Technologies Inc


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

The difference was largest for the chained pipeline (pandas 642.36 ms vs Polars 44.58 ms, 14.41x) and for reading the CSV (630.73 ms vs 44.74 ms, 14.10x). These two are closely linked, because almost all of the pandas pipeline time is the read itself: 642.36 − 630.73 leaves only about 12 ms for the filter, new column, group by and sort combined. My theory is that Polars parses the CSV with multiple threads while pandas' default parser is single-threaded. In the lazy pipeline Polars also reads only the columns it needs (the plan showed PROJECT 2/24), so it avoids building full intermediate DataFrames. The difference was smallest for creating a new column, where pandas won (0.29 ms vs 1.52 ms, 0.19x). A single vectorized subtraction is almost free in pandas, and Polars' fixed cost for building and running an expression is larger than the work itself. Among the operations Polars won, filter had the smallest margin (9.91 ms vs 4.31 ms, 2.30x).

### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

I was surprised that pandas beat Polars on creating a new column (0.29 ms vs 1.52 ms), since I expected Polars to win every row. The lazy test also surprised me: scan_csv(...).filter(...) without .collect() took 0.018 ms because nothing was executed, while with .collect() it took 46.180 ms, which is the real cost. Timing a lazy query without .collect() would make Polars look thousands of times faster than it really is. Another thing I noticed was that pandas raised a DtypeWarning while reading the CSV (column 23, "Unnamed: 22", has mixed types), while Polars loaded the same file without any warning. Finally, the group-by outputs looked different at first glance: pandas returned the categories sorted alphabetically, while Polars returned them in arbitrary order and printed large totals in scientific notation (3.9204e7). The three categories that appeared in both outputs (Blouse, Dupatta, Saree) had identical totals, so the two libraries agree.



### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

Pandas was clearer for simple operations like filtering and adding a column: df[df["Amount"] > threshold] and df.assign(new_col=a - b) read almost like plain English. Polars was harder at first because every operation needs pl.col(...) expressions, and I had to look up .group_by() (named groupby in pandas) and how .agg([...]) takes a list of expressions with .alias(...). Pandas' named aggregation, agg(total=(col, "sum")), was also something I had to look up. Once learned, the Polars pattern was more consistent across filter, new column and group by, and the lazy chain (scan_csv → filter → with_columns → group_by → agg → sort → collect) read top to bottom as one pipeline. The easiest mistake to make was forgetting .collect(), because then nothing runs.


### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

Only partly. I reproduced the "5x or more" claim on the heavy operations: group by (5.49x), sort + top N (6.11x) and join (5.27x), and I exceeded it on reading the file (14.10x) and the chained pipeline (14.41x). I did not reproduce it on filter (2.30x), and new column was the opposite, with pandas about 5x faster (0.19x). Possible reasons the published numbers look larger or more uniform: they usually use multi-gigabyte files, where multithreading pays off and fixed overhead disappears, whereas my dataset has only 128,975 rows and 24 columns. They often time a whole query including the file read, which is where Polars gains most, and they use lazy mode, which pandas has no equivalent for. They also tend to pick operations that parallelize well and run on x86 servers with many cores, while my machine is a Windows 11 laptop with an ARM Qualcomm processor. And they often run pandas with default settings, where the pyarrow engine would narrow the read gap.

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

I would choose pandas for small to medium data (roughly under 100 MB) explored interactively in a notebook, or for projects that depend on libraries built around pandas DataFrames, such as scikit-learn, statsmodels or plotting tools. At that size my results show no real benefit from Polars (new column was 5x slower), and the familiar syntax and ecosystem matter more. I would choose Polars for large datasets (roughly over 1 GB), such as a nightly ETL job that reads big CSV or Parquet files and does heavy filtering, grouping and joins, or for memory-constrained environments. There the multithreading and lazy query optimization gave 5x to 14x in my benchmark.

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

Library: DuckDB
Documentation: https://duckdb.org/docs/

DuckDB is an in-process analytical SQL engine that runs queries directly on CSV and Parquet files and on pandas or Polars DataFrames, with no database server to set up. It exists to give fast, columnar analytics to people who already know SQL, on data that is too large or too slow for plain pandas.

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot